<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
quick template
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "MXX-[Titel]"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace,
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---

In [ ]:
# ── Standardbibliotheken ──────────────────────────────────────────────────
import time
from typing import TypedDict

# ── LangChain / LangGraph ─────────────────────────────────────────────────
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ── Weitere Bibliotheken (nach Bedarf einkommentieren) ────────────────────
# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings
# from langchain_core.documents import Document
# from pydantic import BaseModel, Field
from IPython.display import Image as IPImage, display

In [ ]:
# ── Modell ────────────────────────────────────────────────────────────────
# Modell-Auswahl: siehe Agenten/docs/frameworks/Modell_Auswahl_Guide.md
# Demo/Grundlagen:  init_chat_model(BASELINE, temperature=0.0)
# Worker/Content:   init_chat_model(WORKER)
# Router/Supervisor: init_chat_model(JUDGE)

llm = init_chat_model(BASELINE, temperature=0.0)

# 2 | Chain — `prompt | llm | parser`
---



**Wann verwenden?** Für einzelne, zustandslose LLM-Aufrufe ohne Tools und ohne Gedächtnis.

| Einsatz | Beispiel |
|---------|---------|
| Texttransformation | Zusammenfassen, Übersetzen, Umschreiben |
| Klassifikation | Sentiment, Kategorie, Priorität |
| Extraktion | Structured Output, Entitäten |

> Kein Gedächtnis · Keine Tools · Keine Verzweigung → `chain.invoke()`

In [ ]:
# ── ChatPromptTemplate ────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist ein hilfreicher Assistent. Antworte auf Deutsch."),
    ("human", "{input}"),
])

In [ ]:
# ── Parser  ─────────────────────────────────────────────────────────────
parser = StrOutputParser()

In [ ]:
# ── Chain ───────────────────────────────────────────────────────────────
chain = prompt | llm | parser

In [ ]:
# ── LangSmith config ──────────────────────────────────────────────────────
# run_name: erscheint in LangSmith als Trace-Name
# tags:     ermöglichen Filterung im LangSmith-Dashboard
LANGSMITH_CONFIG = {"run_name": "MXX-[Titel]", "tags": ["mxx"]}

result = chain.invoke({"input": "Hallo!"}, config=LANGSMITH_CONFIG)
mprint(result)

# 3 | Tool — `@tool`
---




**Wann verwenden?** Wenn der Agent externe Aktionen ausführen soll — Berechnungen, APIs, Datenbankzugriff.

| Einsatz | Beispiel |
|---------|---------|
| Berechnung | Mathematik, Währungsrechner, Statistik |
| Externe API | Wetter, Suche, Datenbank, MCP |
| Dateioperationen | Lesen, Schreiben, Verzeichnis |

> Der **Docstring** ist die Anweisung ans LLM — präzise formulieren, das LLM wählt das Tool danach aus.

In [ ]:
# ── Tool-Definition ───────────────────────────────────────────────────────
@tool
def mein_tool(eingabe: str) -> str:
    """[Beschreibung: Was macht dieses Tool? Wird direkt an das LLM übergeben.]"""
    # [Tool-Logik hier]
    return f"Ergebnis: {eingabe}"

# Tools testen
print(mein_tool.invoke({"eingabe": "Test"}))

# Tools-Liste für create_react_agent
tools = [mein_tool]

In [ ]:
# ── Typischer Agenten-Pattern: create_react_agent ─────────────────────────
# LLM wählt Tools selbst — ideal für flexible, Tool-basierte Agenten
react_memory = MemorySaver()
react_agent  = create_react_agent(
    model=llm,
    tools=tools,
    prompt="Du bist ein hilfreicher Assistent. Nutze die verfügbaren Tools. Antworte auf Deutsch.",
    checkpointer=react_memory,
)

react_config = dict(LANGSMITH_CONFIG)
react_config["configurable"] = {"thread_id": f"react-{int(time.time())}"}

react_result = react_agent.invoke(
    {"messages": [("human", "Teste das Tool mit 'Hallo Welt'.")]},
    config=react_config,
)
mprint(f"**Antwort:** {react_result['messages'][-1].content}")

# 4 | StateGraph — strukturierter Agent mit Gedächtnis
---




**Wann welchen Ansatz?**

| Ansatz | Wann? |
|--------|-------|
| `chain` | Eine Anfrage, kein Zustand, keine Tools |
| `create_react_agent` | LLM wählt Tools selbst — flexibel, wenig Code |
| `StateGraph` | Ablauf fest definiert, Verzweigungen, volle Kontrolle |

> **`MemorySaver`** als `checkpointer` → Agent merkt sich den Kontext über mehrere Aufrufe.  
> **`thread_id`** identifiziert die Sitzung — jeder Nutzer / jedes Gespräch bekommt eine eigene ID.

In [ ]:
# ── State ─────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    input:  str
    output: str

# ── Nodes ─────────────────────────────────────────────────────────────────
def node_a(state: AgentState) -> dict:
    """Node A — ruft Tool auf"""
    ergebnis = mein_tool.invoke({"eingabe": state["input"]})
    return {"output": ergebnis}

def node_b(state: AgentState) -> dict:
    """Node B — LLM formuliert finale Antwort auf Basis des Tool-Ergebnisses"""
    response = llm.invoke([HumanMessage(f"Fasse zusammen: {state['output']}")])
    return {"output": response.content}

# ── Graph aufbauen ────────────────────────────────────────────────────────
graph = StateGraph(AgentState)

graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)

graph.add_edge(START,    "node_a")
graph.add_edge("node_a", "node_b")
graph.add_edge("node_b", END)

# ── Compile mit MemorySaver ───────────────────────────────────────────────
memory = MemorySaver()
agent  = graph.compile(checkpointer=memory)

# ── Visualisierung ────────────────────────────────────────────────────────
try:
    display(IPImage(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(Graph-Visualisierung: {e})")

# ── Test-Aufruf mit Sitzungs-ID ───────────────────────────────────────────
session_config = dict(LANGSMITH_CONFIG)
session_config["configurable"] = {"thread_id": f"session-{int(time.time())}"}

result = agent.invoke(
    {"input": "Test-Eingabe", "output": ""},
    config=session_config,
)
mprint(f"**Output:** {result['output']}")

In [ ]:
# ── LangSmith Trace ───────────────────────────────────────────────────────
import time as _t; _t.sleep(2)
show_trace("MXX-[Titel]", limit=3, show_steps=True)

# A | Aufgabe
---



Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

<p><font color='black' size="5">
[Aufgabentitel]
</font></p>

[Aufgabenbeschreibung]

**Teilaufgaben:**
- Teilaufgabe 1
- Teilaufgabe 2